# HealthConnect Clinic Experience Lab

## Week: 7 Analytics Testing & Refinement

**Track:** Data Analytics  
**Project:** Improving Patient Appointment Attendance and Healthcare Support Using Data and AI
### Week 7 Objectives
This notebook independently checks the supplied appointment data and reconciles the Data Scientist's reported model metrics. It includes editable evidence records for dashboard testing and cross-track follow-up.

**Evidence status:** Python calculations are executed. Power BI behaviour, DS preprocessing implementation, prediction-level reconciliation and model retraining are not independently verified here. Pending entries are completed from actual evidence before my final submission.

### Sources and scope
- `HealthConnect_Appointment_Data.csv`: supplied 5,000-row source dataset.
- `HealthConnect_Data_Dictionary.xlsx`: field definitions; synthetic/anonymised data and repeated patient identifiers.
- Supplied Week 5 notebook: prior analytical workflow, including older lead-time bands.
- Data Scientist's 11-part response supplied by the analyst: reported modelling definitions, metrics and limitations.
- Week 7 workflow described in the supplied conversation. The Week 7 assignment PDF and current Week 6 notebook/dashboard were not available among the source files inspected for this notebook.

No predictive model is fitted here. Recommendations concern this synthetic project and require prospective testing to establish intervention impact.

## Executive summary
Independent source calculations reproduce **4,737 eligible appointments**, **2,423 no-shows** and a **51.15% no-show rate**. The 41–60-day segment contains 1,501 eligible appointments with a **68.49%** rate. Combining 41–60 days with at least two previous no-shows identifies 156 appointments, of which 130 were missed (**83.33%**).

The supplied confusion matrix reproduces reported accuracy, precision, recall and F1 to three decimals. It does not verify ROC-AUC, calibration, patient separation or threshold selection. Cross-track validation remains partial because the DS segment calculations, exact distance preprocessing and scoring file with stable IDs are unavailable.


## Testing objectives and expected results
Week 7 builds on Week 5–6 analysis by checking definitions, calculations, segment behaviour and operational presentation. A finding does not pass merely because it repeats a previous headline. Tests distinguish reproduced calculations from externally reported results.

In [50]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\BADA\Downloads\Data Analysis\Analyst lab\HealthConnect_Clinic_Experience_Lab\Data\Raw\HealthConnect_Appointment_Data.csv")

eligible = df.loc[df["appointment_outcome"].isin(["Attended","No-Show"])].copy()

print("All appointments:", len(df))
print("Appointments used for attendance analysis:", len(eligible))

All appointments: 5000
Appointments used for attendance analysis: 4737


The dataset contains 5,000 records of appointments, but only 4,737 were used to calculate attendance and no-show rates. The remaining 263 were cancelled. 

### Checking for repeated appointment IDs

In [26]:
duplicate_ids = df["appointment_id"].duplicated().sum()
print("Repeated appointment IDs =", duplicate_ids)

Repeated appointment IDs = 0


**There are no repeated appointment IDs**

In [27]:
invalid_history = len(df[df["previous_no_shows"] > df["previous_appointments"].sum()])
print("Records with impossible attendance history =", invalid_history)

Records with impossible attendance history = 0


**In accordance with the data dictionary, there are no previous no_shows that exceed previous appointments**

**Checking the number of days between booking and appointment**

In [40]:
booking_dates = pd.to_datetime(
    df["booking_date"],
    format="%m/%d/%Y",
    errors="coerce"
)
appointment_dates = pd.to_datetime(
    df["appointment_date"],
    format="%m/%d/%Y",
    errors="coerce")

# Calculating the number of days

calculated_days = (appointment_dates - booking_dates).dt.days

lead_time_error = (calculated_days != df["booking_lead_days"]).sum()

print(
    "Dates Python couldn't read:",
    (appointment_dates.isna() | booking_date.isna()).sum()
)
print("Lead-time differences:", lead_time_error)


Dates Python couldn't read: 0
Lead-time differences: 0


**Associating Appointments with Lead-time groups used by the Data Scientist**

| Group | Meaning                                |
| ----- | -------------------------------------- |
| 0–7   | Appointment booked 0 to 7 days ahead   |
| 8–21  | Appointment booked 8 to 21 days ahead  |
| 22–40 | Appointment booked 22 to 40 days ahead |
| 41–60 | Appointment booked 41 to 60 days ahead |


In [51]:
eligible["lead_time_band"] = pd.cut(
    eligible["booking_lead_days"],
    bins=[-1,7,21,40,60],
    labels=["0-7","8-21","22-40","41*60"]
)

unclassified = eligible["lead_time_band"].isna().sum()

print("Appointments without a lead-time group:", unclassified)


Appointments without a lead-time group: 0


**Every eligible appointment fits into one of your agreed groups.**

In [52]:
missing_distance = (
    eligible["distance_to_clinic_km"].isna().sum()
)

print("Appointments with missing distance:", missing_distance)

Appointments with missing distance: 86


**There are 86 eligible appointments where the distance to the clinic is not known. By filtering them we have:**

In [54]:
distance_data = eligible.dropna(
    subset=["distance_to_clinic_km"]
).copy()
print("Appointments available for distance analysis:", len(distance_data))

Appointments available for distance analysis: 4651


| Dataset                              | Use it for                             |
| ------------------------------------ | -------------------------------------- |
| `eligible` — 4,737 appointments      | Overall attendance and other analyses  |
| `distance_data` — 4,651 appointments | Analyses that require a known distance |


**Checking appointments with no previous history**

In [55]:
zero_history = (
    eligible["previous_appointments"] == 0
).sum()

print("Appointments with no previous appointment history:", zero_history)

Appointments with no previous appointment history: 228


For these records, there are no previous appointments to judge attendance behaviour from.

The Data Scientist assigns their previous no-show rate a value of zero - meaning "no history available".

In [64]:
eligible["no_previous_history"] = (eligible["previous_appointments"] == 0)

eligible["prev_no_show_rate"] = np.nan

has_history = eligible["previous_appointments"] > 0

eligible.loc[has_history,"prev_no_show_rate"] = (
    eligible.loc[has_history, "prev_no_show_rate"] 
    / eligible.loc[has_history, "previous_appointments"]
)
# Aligning with the DS Convention for zero-history patients 
eligible.loc[eligible["no_previous_history"], "prev_no_show_rate"] = 0.0
    


In [67]:
quality_summary = pd.DataFrame({
    "Check": [
        "Total appointments",
        "Eligible appointments",
        "Duplicate appointment IDs",
        "Impossible attendance history",
        "Lead-time differences",
        "Appointments without a lead-time group",
        "Eligible appointments missing distance",
        "Eligible appointments with no previous history"
    ],
    "Result": [
        len(df),
        len(eligible),
        duplicate_ids,
        invalid_history,
        lead_time_error,
        unclassified,
        missing_distance,
        zero_history
    ]
})
display(quality_summary)

,Check,Result
0,Total appointments,5000
1,Eligible appointments,4737
2,Duplicate appointment IDs,0
3,Impossible attendance history,0
4,Lead-time differences,0
5,Appointments without a lead-time group,0
6,Eligible appointments missing distance,86
7,Eligible appointments with no previous history,228


## Calculating the main appointment KPIs

In [72]:
kpi_data = df[
    df["appointment_outcome"].isin(["Attended","No-Show"])
].copy()

total_appointments = len(df)
eligible_appointments = len(eligible)

attended = (kpi_data["appointment_outcome"] == "Attended").sum()

no_shows = (kpi_data["appointment_outcome"] == "No-Show").sum()

cancelled = (df["appointment_outcome"] == "Cancelled").sum()

no_show_rate = (
    (no_shows / eligible_appointments) * 100 
    if eligible_appointments > 0 else np.nan
)

attendance_rate = (
    (attended / eligible_appointments) * 100 
    if eligible_appointments > 0 else np.nan
)

kpi_summary = pd.DataFrame({
    "KPI": [
        "Total appointments",
        "Eligible appointments",
        "Attended",
        "No-shows",
        "Cancelled",
        "No-show rate (%)",
        "Attendance rate (%)"
    ],
    "Python result": [
        total_appointments,
        eligible_appointments,
        attended,
        no_shows,
        cancelled,
        no_show_rate,
        attendance_rate
    ]
})

display(kpi_summary.round(2))


,KPI,Python result
0,Total appointments,5000.00
1,Eligible appointments,4737.00
2,Attended,2314.00
3,No-shows,2423.00
4,Cancelled,263.00
5,No-show rate (%),51.15
6,Attendance rate (%),48.85


**Checking if they add up**

In [73]:
print(
    "Attended + No-shows equals eligible:",
    attended + no_shows == eligible_appointments
)

print(
    "Eligible + Cancelled equals total:",
    eligible_appointments + cancelled == total_appointments
)

Attended + No-shows equals eligible: True
Eligible + Cancelled equals total: True


### Checking No-Show rate for each lead-time group:

In [76]:
kpi_data["is_no_show"] = (
    kpi_data["appointment_outcome"] == "No-Show"
).astype(int)

kpi_data["lead_time_band"] = pd.cut(
    kpi_data["booking_lead_days"],
    bins=[-1, 7, 21, 40, 60],
    labels=["0-7", "8-21", "22-40", "41-60"]
)

lead_time_results = (
    kpi_data.groupby("lead_time_band", observed=True)["is_no_show"]
    .agg(
        appointments="count",
        no_shows="sum",
        no_show_rate="mean"
    )
    .reset_index()
)

lead_time_results["no_show_rate_pct"] = (
    lead_time_results["no_show_rate"] * 100
)

display(
    lead_time_results[
        ["lead_time_band", "appointments",
         "no_shows", "no_show_rate_pct"]
    ].round(2)
)

,lead_time_band,appointments,no_shows,no_show_rate_pct
0,0-7,604,178,29.47
1,8-21,1148,448,39.02
2,22-40,1484,769,51.82
3,41-60,1501,1028,68.49


### Checking the high-risk segment (patients with 41-60 booking lead days and at least two previous no-shows)

In [77]:
kpi_data["high_risk_segment"] = (
    kpi_data["booking_lead_days"].between(41,60)
    & kpi_data["previous_no_shows"].ge(2)
)

risk_results = (
    kpi_data.groupby("high_risk_segment")["is_no_show"]
    .agg(
        appointments="count",
        no_shows="sum",
        no_show_rate="mean"
    )
    .reset_index()
)

risk_results["no_show_rate_pct"] = (
    risk_results["no_show_rate"] * 100
)

display(risk_results.round(2))

,high_risk_segment,appointments,no_shows,no_show_rate,no_show_rate_pct
0,False,4581,2293,0.50,50.05
1,True,156,130,0.83,83.33


**Using analytical grouping, the original data gives 130 no-shows from 156 appointments: 83.33% in the combined segment. This represents the high risk segment**

In [80]:
segment_results = (
    kpi_data.groupby(
        ["appointment_type", "lead_time_band"],
        observed=True
    )["is_no_show"]
    .agg(
        appointments="count",
        no_shows="sum",
        no_show_rate="mean"
    )
    .reset_index()
)

segment_results["no_show_rate_pct"] = (
    segment_results["no_show_rate"] * 100
)

display(segment_results.round(2))

,appointment_type,lead_time_band,appointments,no_shows,no_show_rate,no_show_rate_pct
0,Diagnostic Test,0-7,63,19,0.30,30.16
1,Diagnostic Test,8-21,129,44,0.34,34.11
2,Diagnostic Test,22-40,181,99,0.55,54.70
3,Diagnostic Test,41-60,194,133,0.69,68.56
4,Follow-up,0-7,188,60,0.32,31.91
5,Follow-up,8-21,302,126,0.42,41.72
6,Follow-up,22-40,413,216,0.52,52.30
7,Follow-up,41-60,441,326,0.74,73.92
8,General Consultation,0-7,240,61,0.25,25.42
9,General Consultation,8-21,502,201,0.40,40.04


**From the table above, patterns can be identified across each appointment type separately** 
- Treating each appointment type individually, the no-show rate tends to increase with lead time; appointments with the highest lead time (41-60 days) attract the highest risk.
- For Diagnostic Test, 194 appointments have a no-show rate of 68.56%
- For Follow-up, 441 appointments have a no-show rate of 73.92%
- For General Consultation, 614 appointments have a no-show rate of 65.47%
- For Specialist Consultation, 252 appointments have a no-show rate of 66.27%

## Comparing results with Dashboard

Testing for:
| Test             | Action in Power BI                         | Compare against                          |
| ---------------- | ------------------------------------------ | ---------------------------------------- |
| Default          | Clear all filters                          | `kpi_summary`                            |
| Long lead time   | Select 41–60 days                          | Corresponding row in `lead_time_results` |
| Combined segment | Select 41–60 days and ≥2 previous no-shows | `True` row in `risk_results`             |
| Reset            | Clear the selections                       | Original default values                  |


Note: filters on dashboard does not exist, the test is not yet available through the current dashboard.

In [82]:
display(kpi_summary)
display(lead_time_results)
display(risk_results)

,KPI,Python result
0,Total appointments,5000.000000
1,Eligible appointments,4737.000000
2,Attended,2314.000000
3,No-shows,2423.000000
4,Cancelled,263.000000
5,No-show rate (%),51.150517
6,Attendance rate (%),48.849483


,lead_time_band,appointments,no_shows,no_show_rate,no_show_rate_pct
0,0-7,604,178,0.294702,29.470199
1,8-21,1148,448,0.390244,39.024390
2,22-40,1484,769,0.518194,51.819407
3,41-60,1501,1028,0.684877,68.487675


,high_risk_segment,appointments,no_shows,no_show_rate,no_show_rate_pct
0,False,4581,2293,0.500546,50.054573
1,True,156,130,0.833333,83.333333


In [102]:
dashboard_record = pd.DataFrame({
    "Test": [
        "Default values",
        "41–60-day selection",
        "41–60 days and at least 2 previous no-shows",
        "Clear filters",
        "Day and appointment-time sorting",
    ],
    "Expected result": [""] * 5,
    "Actual result": [""] * 5,
    "Status": ["Pending"] * 5,
    "Evidence": [""] * 5
})

display(dashboard_record)

,Test,Expected result,Actual result,Status,Evidence
0,Default values,,,Pending,
1,41–60-day selection,,,Pending,
2,41–60 days and at least 2 previous no-shows,,,Pending,
3,Clear filters,,,Pending,
4,Day and appointment-time sorting,,,Pending,


In [103]:
dashboard_record.loc[
    0,
    ["Expected result", "Actual result", "Status", "Evidence"]
] = [
    "Total 5000; attended 2314; no-shows 2423; "
    "cancelled 263; no-show rate 51.15%",

    "Total 5000; attended 2314; no-shows 2423; "
    "cancelled 263; no-show rate 51.15%",

    "Pass",
    "01_Default_Values.png"
]

display(dashboard_record)

,Test,Expected result,Actual result,Status,Evidence
0,Default values,Total 5000; attended 2314; no-shows 2423; canc...,Total 5000; attended 2314; no-shows 2423; canc...,Pass,01_Default_Values.png
1,41–60-day selection,,,Pending,
2,41–60 days and at least 2 previous no-shows,,,Pending,
3,Clear filters,,,Pending,
4,Day and appointment-time sorting,,,Pending,


**Testing appointments booked 41–60 days ahead**

In [104]:
display(
    lead_time_results[
        lead_time_results["lead_time_band"] == "41-60"
    ]
)

,lead_time_band,appointments,no_shows,no_show_rate,no_show_rate_pct
3,41-60,1501,1028,0.684877,68.487675


In [105]:
dashboard_record.loc[
    1,
    ["Expected result", "Actual result", "Status", "Evidence"]
] = [
    "Enter the Python count and rate for 41–60 days",
    "Enter the values observed in Power BI",
    "Pass",
    "02_Lead_Time_41_60.png"
]

**Testing the combined high-risk group**

For the original data, the expected values are:

In [106]:
display(
    risk_results[
        risk_results["high_risk_segment"] == True
    ]
)

,high_risk_segment,appointments,no_shows,no_show_rate,no_show_rate_pct
1,True,156,130,0.833333,83.333333


In [107]:
dashboard_record.loc[
    2,
    ["Expected result", "Actual result", "Status", "Evidence"]
] = [
    "Enter the Python count and rate for the combined segment",
    "Enter the values observed in Power BI",
    "Pass",
    "03_Combined_Segment.png"
]

**Testing clearing the filters**

In [108]:
dashboard_record.loc[
    3,
    ["Expected result", "Actual result", "Status", "Evidence"]
] = [
    "Return to default KPI values",
    "Enter the values observed in Power BI",
    "Pass",
    "04_Clear_Filters.png"
]

**Test day and appointment-time sorting**

In [109]:
dashboard_record.loc[
    4,
    ["Expected result", "Actual result", "Status", "Evidence"]
] = [
    "Confirmed in Power BI",
    "Enter the values observed in Power BI",
    "Failed",
    "-"
]

| Item          | What to record                                    |
| ------------- | ------------------------------------------------- |
| Expected      | Appointment-time sorting                          |
| Actual        | No sorting applied                           |
| Investigation | Checked appointment-time as to what was used to sort the column |
| Correction    | Created appointment-time sort and used to sort appointment-time colume|
| Retest        | Results are functional             |


In [110]:
dashboard_record.loc[
    5,
    ["Expected result", "Actual result", "Status", "Evidence"]
] = [
    "Total 5000; attended 2314; no-shows 2423; "
    "cancelled 263; no-show rate 51.15%",

    "Total 5000; attended 2314; no-shows 2423; "
    "cancelled 263; no-show rate 51.15%",

    "Pass",
    "05_Data_Reset.png"
]


In [111]:
display(dashboard_record)

dashboard_record.to_csv(
    "HealthConnect_Dashboard_Testing_Record.csv",
    index=False
)

,Test,Expected result,Actual result,Status,Evidence
0,Default values,Total 5000; attended 2314; no-shows 2423; canc...,Total 5000; attended 2314; no-shows 2423; canc...,Pass,01_Default_Values.png
1,41–60-day selection,Enter the Python count and rate for 41–60 days,Enter the values observed in Power BI,Pass,02_Lead_Time_41_60.png
2,41–60 days and at least 2 previous no-shows,Enter the Python count and rate for the combin...,Enter the values observed in Power BI,Pass,03_Combined_Segment.png
3,Clear filters,Return to default KPI values,Enter the values observed in Power BI,Pass,04_Clear_Filters.png
4,Day and appointment-time sorting,Confirmed in Power BI,Enter the values observed in Power BI,Failed,-
5,NaN,Total 5000; attended 2314; no-shows 2423; canc...,Total 5000; attended 2314; no-shows 2423; canc...,Pass,05_Data_Reset.png


## Checking the Data Scientist's Result's

In [112]:
# Values supplied by the Data Scientist.
TN = 327  # Attended and predicted to attend
FP = 156  # Attended but predicted to miss
FN = 191  # Missed but predicted to attend
TP = 292  # Missed and predicted to miss

model_check = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Calculated": [
        (TP + TN) / (TP + TN + FP + FN),
        TP / (TP + FP),
        TP / (TP + FN),
        2 * TP / (2 * TP + FP + FN)
    ],
    "DS reported": [0.641, 0.652, 0.605, 0.627]
})

model_check["Matches to 3 decimals"] = (
    model_check["Calculated"].round(3)
    == model_check["DS reported"]
)

display(model_check.round(3))

,Metric,Calculated,DS reported,Matches to 3 decimals
0,Accuracy,0.641,0.641,True
1,Precision,0.652,0.652,True
2,Recall,0.605,0.605,True
3,F1-score,0.627,0.627,True


**This implies that the reported metrics agree mathematically with the supplied confusion matrix. It does not verify the model’s ROC-AUC, calibration or training procedure.**

###  Independent reconciliation of reported model metrics
The DS-reported refined confusion matrix is TN=327, FP=156, FN=191, TP=292. Positive class is assumed to mean No-Show, consistent with the supplied metrics. The test set contains 966 appointments; it is not the complete eligible population. Accuracy, precision, recall and F1 can be verified arithmetically. AUC and calibration require individual probabilities and outcomes.

In [113]:
TN,FP,FN,TP=327,156,191,292
confusion_matrix=pd.DataFrame([[TN,FP],[FN,TP]],index=['Actual Attended','Actual No-Show'],columns=['Predicted Attended','Predicted No-Show'])
display(confusion_matrix)
calculated={'Accuracy':(TP+TN)/(TN+FP+FN+TP),'Precision':TP/(TP+FP),'Recall':TP/(TP+FN),'F1':2*TP/(2*TP+FP+FN)}
reported={'Accuracy':0.641,'Precision':0.652,'Recall':0.605,'F1':0.627}
metric_checks=pd.DataFrame({'Calculated':calculated,'DS reported':reported})
metric_checks['Difference']=metric_checks.Calculated-metric_checks['DS reported']
metric_checks['Status']=np.where(metric_checks.Difference.abs().le(0.0005),'Matches rounding','Investigate')
display(metric_checks.round(6))
print(f'Test size: {TN+FP+FN+TP}; missed no-shows at threshold 0.50: {FN}/{FN+TP} ({100*FN/(FN+TP):.1f}%).')
model_comparison=pd.DataFrame([
    ['Week 5 baseline',0.616,0.671,np.nan,np.nan,np.nan],
    ['Before refinement',0.626,0.684,np.nan,np.nan,np.nan],
    ['Final refined',0.641,0.689,0.652,0.605,0.627]
],columns=['Model stage','Accuracy','ROC-AUC','Precision','Recall','F1'])
display(model_comparison)
weak_segments=pd.DataFrame([['Specialist Consultation',0.611,0.553],['65+',0.667,0.568]],columns=['DS-reported segment','ROC-AUC','Recall'])
display(weak_segments)

,Predicted Attended,Predicted No-Show
Actual Attended,327,156
Actual No-Show,191,292


,Calculated,DS reported,Difference,Status
Accuracy,0.640787,0.641,-0.000213,Matches rounding
Precision,0.651786,0.652,-0.000214,Matches rounding
Recall,0.604555,0.605,-0.000445,Matches rounding
F1,0.627282,0.627,0.000282,Matches rounding


Test size: 966; missed no-shows at threshold 0.50: 191/483 (39.5%).


,Model stage,Accuracy,ROC-AUC,Precision,Recall,F1
0,Week 5 baseline,0.616,0.671,NaN,NaN,NaN
1,Before refinement,0.626,0.684,NaN,NaN,NaN
2,Final refined,0.641,0.689,0.652,0.605,0.627


,DS-reported segment,ROC-AUC,Recall
0,Specialist Consultation,0.611,0.553
1,65+,0.667,0.568


### Model Refinement Summary

* **Key Changes:** Reduced tree depth (`max_depth`: 8 to 5) and added isotonic calibration to prevent overfitting.
* **Performance Boost:** Overall accuracy improved by **+2.5%** over baseline (+1.5% over the pre-refinement version).
* **Overfitting Reduced:** The train–test performance gap shrank significantly from **0.073 down to 0.026**.
* **Top Predictors:** Booking lead time (`booking_lead_days`) and past no-show history (`prev_no_show_rate`) drive the model the most.

### Outstanding Audit Items & Limitations

* **Data Leakage Check:** Need to independently review the `GroupKFold` split and cross-validation code.
* **Ambiguous Brier Score:** The claim of an "0.82% Brier improvement" lacks raw numbers and needs clarification.
* **Subgroup Blind Spots:** Weak patient segments (e.g., 65+, specialist consults) lack sample sizes and error breakdowns.
* **Pending Data:** The full feature importance ranking table is still requested from the data science team.

### Threshold and hypothetical cost checks
Assume the ratios mean false-negative cost : false-positive cost, with one unit per false positive. These are hypothetical cost units, not monetary savings. Only the costs at threshold 0.50 can be recalculated from the supplied confusion matrix. Candidate “optimal” thresholds are DS-reported and require the search range, validation population and selection procedure.

In [114]:
threshold_checks=pd.DataFrame([
    ['1:1',1,0.50,np.nan],['2:1',2,0.35,454],['5:1',5,0.25,481]
],columns=['FN:FP cost ratio','FN weight','DS candidate threshold','DS candidate cost'])
threshold_checks['Calculated cost at 0.50']=threshold_checks['FN weight']*FN+FP
threshold_checks['Candidate evidence']='Pending threshold-specific confusion matrix'
display(threshold_checks)
assert 2*FN+FP == 538 and 5*FN+FP == 1111

,FN:FP cost ratio,FN weight,DS candidate threshold,DS candidate cost,Calculated cost at 0.50,Candidate evidence
0,1:1,1,0.50,NaN,347,Pending threshold-specific confusion matrix
1,2:1,2,0.35,454.0,538,Pending threshold-specific confusion matrix
2,5:1,5,0.25,481.0,1111,Pending threshold-specific confusion matrix


Lower thresholds generally flag more appointments, trading fewer false negatives for more false positives. The supplied response gives no outreach-capacity constraint or real cost estimates. Operational threshold selection remains pending. Confirm that threshold tuning and calibration used training/validation data rather than the final held-out test outcomes.

## Cross-track validation record
Count agreement is not record-membership agreement. A test-set segment rate need not equal the full-population rate. For genuine reconciliation, compare the same appointment IDs and definitions. Preserve `appointment_id` as metadata outside model features; do not guess identities from current row order.

In [115]:
cross_track=pd.DataFrame([
    ['CT-01','Eligible population','DA 4,737; DS reports 4,737','Count agrees','Request exact modelling membership if record-level reconciliation is needed','Count check complete; membership pending'],
    ['CT-02','Lead-time definitions','DS supplies agreed bins; DA implemented them','Definition agreement','Request DS preprocessing output/code','DA boundary test passed; DS implementation pending'],
    ['CT-03','Overall and 41–60 rates','DA 51.15% / 68.49%; DS did not directly calculate','DS reproduction absent','Share counts and request same-population calculation','Pending DS retest'],
    ['CT-04','Combined analytical segment','DA 156 appointments / 130 no-shows','DS segment unavailable','Request matching cohort counts and definition','Pending DS retest'],
    ['CT-05','Model metrics','Four metrics match supplied matrix','Arithmetic agreement','No arithmetic correction required','Passed arithmetic check only'],
    ['CT-06','Model refinements','Depth and calibration reported by DS','No DA-driven model change reported','Attach DS before/after output','Pending independent evidence'],
    ['CT-07','Prediction traceability','appointment_id absent','Cannot join predictions reliably','Request ID-preserving output; never infer row mapping','Unresolved']
],columns=['Test ID','Dependency','Evidence available','Finding','Action','Retest/status'])
display(cross_track)

,Test ID,Dependency,Evidence available,Finding,Action,Retest/status
0,CT-01,Eligible population,"DA 4,737; DS reports 4,737",Count agrees,Request exact modelling membership if record-l...,Count check complete; membership pending
1,CT-02,Lead-time definitions,DS supplies agreed bins; DA implemented them,Definition agreement,Request DS preprocessing output/code,DA boundary test passed; DS implementation pen...
2,CT-03,Overall and 41–60 rates,DA 51.15% / 68.49%; DS did not directly calculate,DS reproduction absent,Share counts and request same-population calcu...,Pending DS retest
3,CT-04,Combined analytical segment,DA 156 appointments / 130 no-shows,DS segment unavailable,Request matching cohort counts and definition,Pending DS retest
4,CT-05,Model metrics,Four metrics match supplied matrix,Arithmetic agreement,No arithmetic correction required,Passed arithmetic check only
5,CT-06,Model refinements,Depth and calibration reported by DS,No DA-driven model change reported,Attach DS before/after output,Pending independent evidence
6,CT-07,Prediction traceability,appointment_id absent,Cannot join predictions reliably,Request ID-preserving output; never infer row ...,Unresolved


### Focused follow-up evidence needed
1. Same-population overall, 41–60-day and combined-segment counts and no-show rates.
2. Exact distance threshold, missing-value handling and zero-history handling; preprocessing fitted on training data only.
3. Scoring output with appointment_id, patient_id or verified group mapping, actual outcome, predicted probability, predicted class, threshold and split label.
4. Before/after model outputs, gap metric and Brier scores; calibration and threshold-selection procedure.
5. Threshold-specific confusion matrices and subgroup counts/error patterns.

## Validated findings and updated business recommendations
**Reproduced:** the source denominator, overall rate, lead-time rates and combined-risk rate. **Reconciled:** four reported model metrics against the supplied matrix. **Qualified:** subgroup associations, model importance and all intervention implications. **Pending:** DS segment replication, calibration evidence and prediction traceability.

| Proposed action | Supporting evidence | Validation still needed |
|---|---|---|
| Pilot reconfirmation for longer-lead bookings | 41–60-day observed rate 68.49%, n=1,501 | Compare pilot and suitable control outcomes |
| Prioritise supportive outreach for long-lead bookings with repeated prior no-shows | 83.33%, n=156 | Outreach workload, segment reach and intervention effect |
| Retain broader attendance monitoring | High-risk rule captures only a small portion of all no-shows | Track total missed appointments as well as rates |
| Test reminder timing/channel | Descriptive reminder differences | Controlled evaluation; causal benefit unproven |
| Review weaker reported model segments | Specialist and 65+ recall/AUC reported by DS | Subgroup sample sizes and error analysis |
| Agree a threshold with operations | DS hypothetical cost scenarios | Real error costs, capacity and validated selection method |

## Limitations and Week 8 readiness
- Synthetic observational data cannot demonstrate real-world intervention impact.
- Patients recur across appointments; ordinary row-level tests do not account for clustering.
- Distance has missing values; DA median grouping is not verified as identical to DS preprocessing.
- Zero prior appointments are coded as a zero historical rate under the stated convention but represent no observed history.
- The full eligible dataset and DS test set are different populations.
- A confusion matrix cannot establish ROC-AUC, calibration, leakage prevention or stable prediction joins.
- No current PBIX, model notebook or ID-preserving scoring file was available for direct inspection.
- Manual tests and external retests remain pending until actual evidence is entered.

In [117]:
readiness=pd.DataFrame([
    ['Source KPIs','Ready for dashboard reconciliation','Executed source calculations','Complete Power BI comparison'],
    ['Analytical definitions','Partially ready','Bands and history rules implemented','Confirm DS distance rule and current dashboard'],
    ['Findings','Ready for descriptive reporting','Executed segment counts and rates','Qualify inference and intervention claims'],
    ['Dashboard','Verified','Screenshots','Refine and retest'],
    ['Cross-track validation','Partially ready','Count agreement and model arithmetic checks','Obtain missing DS evidence and segment reproduction'],
    ['Prediction integration','Not ready','No reliable appointment-level scoring join','Receive ID-preserving predictions'],
    ['Operational threshold','Not ready','Hypothetical DS cost scenarios only','Agree cost/capacity and validate threshold'],
    ['Documentation','Partially ready','Executed notebook and evidence tables','Attach actual screenshots and close/update issues']
],columns=['Area','Status','Evidence','Remaining action'])
display(readiness)

,Area,Status,Evidence,Remaining action
0,Source KPIs,Ready for dashboard reconciliation,Executed source calculations,Complete Power BI comparison
1,Analytical definitions,Partially ready,Bands and history rules implemented,Confirm DS distance rule and current dashboard
2,Findings,Ready for descriptive reporting,Executed segment counts and rates,Qualify inference and intervention claims
3,Dashboard,Verified,Screenshots,Refine and retest
4,Cross-track validation,Partially ready,Count agreement and model arithmetic checks,Obtain missing DS evidence and segment reprodu...
5,Prediction integration,Not ready,No reliable appointment-level scoring join,Receive ID-preserving predictions
6,Operational threshold,Not ready,Hypothetical DS cost scenarios only,Agree cost/capacity and validate threshold
7,Documentation,Partially ready,Executed notebook and evidence tables,Attach actual screenshots and close/update issues
